# Funamizu, Kuhn & Doya (2016) on DANDI:001605

Paper: Funamizu A, Kuhn B, Doya K (2016). Neural substrate of dynamic Bayesian inference in the
cerebral cortex. *Nature Neuroscience* 19, 1682-1689, doi:10.1038/nn.4390.

Dataset: [DANDI:001605](https://dandiarchive.org/dandiset/001605/0.251007.0553), version
0.251007.0553 (doi:10.48324/dandi.001605/0.251007.0553). The paper cites no dandiset version
(it predates the dandiset). This is the only published version and the most recent one; the
draft has the same 94 assets and size and was last modified at publication, so it is pinned.

Claims file: `claims/001605/10.1038-nn.4390.yaml`, written from the version of record (main
text, figure legends, Online Methods and the supplement). The claims file fixes every claim,
its expected values and its criterion; method details it lacks are taken from the paper and
cited where they are used. Readings named O1 to O9 are those of the claims file's header.

| Claim | Tested | Why not |
|---|---|---|
| session-counts, neuron-counts, behaviour-timing, lick-goal-correlation | yes | |
| fig1c-licking-without-sound, supp1-licking-speed | yes | |
| fig1d-muscimol-correlation, fig1e-muscimol-variance, fig1fg-muscimol-no-sound-licking, supp2d-muscimol-overrun, muscimol-locomotion | no | blocked, data missing: the muscimol mice are not in the dandiset |
| sound-responsive-layer3 | no | blocked, data missing: no sound-mapping trials in the files |
| neuron-classes, fig2d-task-relevant-proportions, fig2d-goal-distance-ppc-more, fig2d-after-vs-before, fig2d-layer-distance-preference | yes | |
| fig3b-distance-over-time | no | blocked, too large: the regressions with up to three Gaussian bases over 11 variable sets, estimated at 14 h |
| supp6-distance-not-sound, tuning-similar-ppc-pm | yes | |
| fig3c-continuous-ramp, fig3c-ppc-ramp-without-sound, supp8a-pm-no-sound, supp8b-slopes-lower-without-sound, supp8c-ppc-steeper-than-pm, supp9-preserved-representation | yes | |
| fig4-above-chance, fig4c-ppc-lower-error, fig4d-map-near-goal, fig4e-ppc-lower-uncertainty, supp11-sound-reduces-error | yes | |
| fig5a-map-follows-continuous, fig5a-map-decreases-without-sound, fig5b-map-slope-lower-without-sound, fig5-layer3-steeper-than-layer2, supp13-map-speed, fig5cd-uncertainty-updating, fig5c-uncertainty-decreases-with-sound, lognormal-decoder-better | yes | |
| svm-correlations | no | blocked, too large: leave-one-trial-out SVMs, estimated at 1.4 h per reading |

Method details used throughout and where they come from are in the docstring of `funamizu.py`.
Two choices forced by the data are standard ones: the files hold a lick frequency (1/inter-lick
interval, capped at 10 Hz, held between licks) instead of lick times, so a lick count is the
integral of that frequency and a lick onset is a change of it; and distance-aligned activity is
interpolated at 0.42 cm bin centres.

## Loading

Each file is streamed once and reduced, in four worker processes, to its trial table, behaviour per trial, per-neuron statistics of the classification and the decoder's inputs (`funamizu.process`).

In [ ]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
import time as _time
import numpy as np
import pandas as pd
import funamizu as F
ASSETS = F.assets()
print(f"dandiset {F.DANDISET} version {F.VERSION}: {len(ASSETS)} files, {sum(a[2] for a in ASSETS) / 1e9:.1f} GB")
t0 = _time.time()
with ProcessPoolExecutor(4, mp_context=mp.get_context("spawn")) as ex:
    R = list(ex.map(F.process, [a[0] for a in ASSETS], [a[1] for a in ASSETS]))
print(f"streamed and reduced {len(R)} sessions in {(_time.time() - t0) / 60:.1f} min")

In [ ]:
import numpy as np, pandas as pd
from sklearn.cluster import KMeans
LT = np.log10(F.P_TASK)
sessions = pd.DataFrame([dict(i=i, **r["meta"]) for i, r in enumerate(R)])
sessions["group"] = sessions.area + " L" + sessions.layer.astype(str)
GROUPS = ["PPC L2", "PPC L3", "PPC L5", "PM L2", "PM L3", "PM L5"]

def neuron_table():
    rows = []
    for i, r in enumerate(R):
        ns = r["neurons"]; n = len(ns["peak_after"])
        d = dict(session=np.full(n, i), roi=np.arange(n), area=r["meta"]["area"], layer=r["meta"]["layer"],
                 peak_after=ns["peak_after"], dist_pref=ns["dist_pref"])
        for k, v in ns.items():
            if k.startswith("lp_"):
                d[k] = v
        rows.append(pd.DataFrame(d))
    t = pd.concat(rows, ignore_index=True)
    t["group"] = t.area + " L" + t.layer.astype(str)
    return t

NT = neuron_table()
PRE16 = np.vstack([r["neurons"]["pre16"] for r in R])
COND = np.concatenate([r["neurons"]["cond"] for r in R])          # neurons x 3 conditions x 160 bins (goal distance ascending)
COND_PREF = np.vstack([r["neurons"]["cond_pref"] for r in R])
READINGS = [(b, s, k) for b in "ab" for s in ("67", "33") for k in "ab"]   # O1 baseline, O2 start window, O3 k-means feature

def classify(base, sw, km, seed=0):
    """Classes under one reading of O1 (base), O2 (sw) and O3 (km)."""
    t = NT
    goal_lp = np.minimum(t[f"lp_goal_{base}"], t[f"lp_dist_{base}"])
    lick_lp = t[f"lp_lick_{base}"]
    start_sig = (t[f"lp_start_{base}"] < LT) | (t[f"lp_sdist{sw}_{base}"] < LT)
    goal_sig = goal_lp < LT
    lick_sig = lick_lp < LT
    goal_cls = goal_sig & ~(lick_sig & (lick_lp < goal_lp))          # the more significant alignment wins
    lick_cls = lick_sig & ~goal_cls
    task = start_sig | goal_sig | lick_sig
    before = goal_cls & ~t.peak_after
    aft = (goal_cls & t.peak_after).values
    feat = PRE16[aft] if km == "a" else PRE16[aft].mean(1, keepdims=True)
    lab = KMeans(2, n_init=20, random_state=seed).fit_predict(np.nan_to_num(feat))
    hi = np.argmax([np.nanmean(PRE16[aft][lab == c]) for c in (0, 1)])
    after = np.zeros(len(t), bool); after[np.flatnonzero(aft)[lab == hi]] = True
    reward = np.zeros(len(t), bool); reward[np.flatnonzero(aft)[lab != hi]] = True
    return pd.DataFrame(dict(task=task.values, goal_sig=goal_sig.values, lick=lick_cls.values, before=before.values,
                             after=after, reward=reward, gd=before.values | after))

CLS = {rd: classify(*rd) for rd in READINGS}
CLS_GD = {rd: CLS[rd] for rd in READINGS if rd[1] == "67"}           # O2 does not change goal-distance classes

In [ ]:
from scipy import stats
import pandas as pd
def valid(r):
    return r["trials"]["valid"]
COND_NAMES = ["continuous", "intermittent 1", "intermittent 2"]

## Data check

What was loaded, and each check of the description against the data. The trial table's
`goal_distance_cm` is the start-to-goal distance; `VirtualMovement/move_scan` column 0 is the
forward position within the trial in units of 36/430 cm (it starts at 0 in every trial and
reaches the trial's goal distance when the goal is reached); `Speed/speed_x` is forward speed
in m/s after its conversion factor; `Licking/lick_rate` is a lick frequency in Hz. Imaging and
behaviour share one 30.9 Hz clock: every series has the same number of frames, and the trial
table's `frames` sum to that number. The behaviour streams follow the trial clock: the reward
follows the frame where the position passes the goal by tens of milliseconds, and the trial
records end about 8 s after the reward (the paper's ITI is 7 s). The imaging follows it too:
the decoder, trained on dR/R against the goal distance of each frame, estimates distance far
better than the flat prior (claim fig4-above-chance), which an offset between the clocks would
prevent. The description says the files hold "sound on/off indicators"; the data hold only the
start tone. `sound_on_off` is 1 in exactly one frame per trial, the first, and every row of
`speaker_presentations` starts at a trial start and lasts one frame. Sound and no-sound zones
are therefore computed from the condition and the distance, as the Methods define them.

In [ ]:
print("files:", len(R), "| sessions per mouse:", sessions.groupby("subject").size().to_dict())
print("sessions per area and layer:", sessions.groupby("group").size().reindex(GROUPS).to_dict())
print("ROIs per area and layer:", sessions.groupby("group").n_rois.sum().reindex(GROUPS).to_dict(), "total", int(sessions.n_rois.sum()))
print("ROIs per session:", sessions.n_rois.min(), "to", sessions.n_rois.max(), "| frames per session:", sessions["T"].min(), "to", sessions["T"].max())
ntr = [len(r["trials"]["gd"]) for r in R]
print("trials per file:", set(ntr), "| valid trials (reward within 20 s):", sum(int(r["trials"]["valid"].sum()) for r in R), "of", sum(ntr))
st = np.array([np.bincount(r["trials"]["state"], minlength=3) for r in R]).sum(0)
print("trials per condition (continuous, intermittent 1, intermittent 2):", st.tolist(), "shares", np.round(st / st.sum(), 3).tolist(), "(paper 70, 15, 15%)")
gd = np.concatenate([r["trials"]["gd"] for r in R])
print(f"start-to-goal distance: {gd.min():.2f} to {gd.max():.2f} cm (paper 67 to 134 cm)")
print("dates unique per mouse:", sessions.groupby(["subject", "date"]).size().max() == 1)
# recordings deposited twice: identical trial sequences or identical frame and ROI counts
sig = [tuple(np.round(r["trials"]["gd"], 6)) for r in R]
print("distinct trial sequences:", len(set(sig)), "| distinct (frames, ROIs) pairs:", len(set(zip(sessions["T"], sessions.n_rois))))
rg = pd.DataFrame([dict(pos_min=r["ranges"]["pos"][0], pos_max=r["ranges"]["pos"][1], speed_p1=r["ranges"]["speed"][0], speed_p99=r["ranges"]["speed"][1],
                        lick_max=r["ranges"]["lick"][1], dff_p01=r["ranges"]["dff"][0], dff_p999=r["ranges"]["dff"][1],
                        dff_mean_absmax=r["ranges"]["dff_session_mean_absmax"]) for r in R])
print("ranges over sessions (cm, cm/s, Hz, dR/R):"); print(rg.describe().loc[["min", "max"]].round(4).to_string())
print("frames with sound_on_off = 1 per session:", sorted(set(r["sound_frames"] for r in R)), "| of these at a trial's first frame:", sorted(set(r["sound_at_start"] for r in R)))
print("speaker_presentations rows per session:", min(r["meta"]["n_speaker_rows"] for r in R), "to", max(r["meta"]["n_speaker_rows"] for r in R))
# clock: reward follows reaching the goal by a few frames in every session (behaviour streams on the trial clock)
lag = np.concatenate([(r["trials"]["rew_f"] - r["trials"]["goal_f"])[r["trials"]["valid"]] for r in R]) / F.FS
print(f"reward minus goal-reaching time: median {np.median(lag) * 1000:.0f} ms, 5th to 95th percentile {np.percentile(lag, 5) * 1000:.0f} to {np.percentile(lag, 95) * 1000:.0f} ms")
iti = np.concatenate([((r["trials"]["s0"] + r["trials"]["nfr"]) - r["trials"]["rew_f"])[r["trials"]["valid"]] for r in R]) / F.FS
print(f"trial record end minus reward: {iti.min():.2f} to {iti.max():.2f} s (ITI 7 s in the paper)")
# dR/R against the ratio R in one file: dR/R = (R - mean R) / mean R over the session
p0, a0, _ = min(ASSETS, key=lambda a: a[2])
with F.open_asset(a0) as f:
    Rr = f["processing/ophys/Fluorescence/RawFluorescence/data"][:]
    Dd = f["processing/ophys/DfOverF/dF_over_F/data"][:]
print(f"{p0}: max |(R - mean R)/mean R - dF_over_F| = {np.abs((Rr - Rr.mean(0)) / Rr.mean(0) - Dd).max():.2e}; R mean {Rr.mean():.3f}")
del Rr, Dd

## session-counts

> Imaging data come from 94 sessions of 8 mice (2, 10, 5, 14, 15, 15, 12 and 21 sessions for mice 1 to 8); PPC layers 2, 3 and 5 were imaged in 17, 15 and 14 sessions and PM layers 2, 3 and 5 in 16, 15 and 17.

Source: Online Methods, Behavioral analysis; Results, Goal-distance representation in PPC and PM, paragraph 1.

In [ ]:
per_mouse = sessions.groupby("subject").size()
print("sessions:", len(sessions), "| mice:", sessions.subject.nunique())
print("sessions per mouse, sorted:", sorted(per_mouse.tolist()), "| paper, sorted:", sorted([2, 10, 5, 14, 15, 15, 12, 21]))
print("sessions per mouse by label:", per_mouse.to_dict())
print("sessions per area and layer:", {g: int((sessions.group == g).sum()) for g in GROUPS}, "| paper 17, 15, 14, 16, 15, 17")

**Result: Reproduced.** 94 sessions of 8 mice; the sessions per mouse match the paper's list (2, 5,
10, 12, 14, 15, 15, 21, and the dataset's mouse labels follow the paper's numbering), and
the sessions per area and layer are 17, 15, 14 (PPC) and 16, 15, 17 (PM), all as reported.

## neuron-counts

> 41,031 neurons were imaged, 8,371, 7,108 and 4,859 in PPC layers 2, 3 and 5 and 7,788, 7,005 and 5,900 in PM; these are active cells, 99.6% of the manually selected ROIs.

Source: Results, Goal-distance representation in PPC and PM, paragraph 1; Supplementary Table 1.

Reading (a) of the criterion, all ROIs in the files. Reading (b) is not needed, since (a)
decides the criterion ("under one reading").

In [ ]:
cnt = {g: int(sessions.n_rois[sessions.group == g].sum()) for g in GROUPS}
print("reading (a), all ROIs:", cnt, "total", sum(cnt.values()))
print("paper: 8371, 7108, 4859, 7788, 7005, 5900, total 41031")
print("per-session means:", {g: round(sessions.n_rois[sessions.group == g].mean(), 1) for g in GROUPS}, "| Supplementary Table 1: 492, 474, 347, 487, 467, 347")

**Result: Reproduced.** The files hold 41,031 ROIs, 8,371, 7,108 and 4,859 in PPC layers 2, 3, 5 and
7,788, 7,005 and 5,900 in PM, equal to the paper's counts of active neurons. The files hold
the active ROIs only.

## behaviour-timing

> Mice got the reward 7.4 ± 0.1 s after the trial start and spent 860 ± 10 ms in each no-sound zone of the intermittent conditions (mean ± s.e.m. over 94 sessions).

Source: Online Methods, Behavioral training, paragraph 5.

Time from the trial start (the first frame of the trial record, where the start tone is
marked) to the first reward edge, per valid trial, mean per session, mean over sessions. No-sound
zone time under the criterion's two readings.

In [ ]:
rs = np.array([np.mean(r["trials"]["reward_s"][valid(r)]) for r in R])
za = np.array([r["zone_a"].mean() for r in R]); zb = np.array([r["zone_b"].mean() for r in R])
print(f"start tone to reward: mean over 94 sessions {rs.mean():.3f} s, s.e.m. {stats.sem(rs):.3f} (expected 7.4 +- 0.1)")
print(f"  per-session means range {rs.min():.2f} to {rs.max():.2f} s")
print(f"time per no-sound zone, reading (a) complete zones before the goal: {za.mean():.3f} +- {stats.sem(za):.3f} s")
print(f"time per no-sound zone, reading (b) every zone entered, to the reward: {zb.mean():.3f} +- {stats.sem(zb):.3f} s (expected 0.86 +- 0.01)")
# variants, shown only: start to goal, and from first movement (position > 0) to reward
tg = np.array([np.mean(((r["trials"]["goal_f"] - r["trials"]["s0"]) / F.FS)[valid(r)]) for r in R])
print(f"variant, start tone to reaching the goal: {tg.mean():.3f} +- {stats.sem(tg):.3f} s")
print(f"variant, end of the 970 ms start tone to reward: {(rs - 0.97).mean():.3f} s")
for subj, ix in sessions.groupby("subject").groups.items():
    print(f"  {subj}: reward {rs[ix].mean():.2f} s, zone (a) {za[ix].mean():.3f}, (b) {zb[ix].mean():.3f}, sessions {len(ix)}")
excl = np.array([1 - valid(r).mean() for r in R])
print(f"trials excluded (over 20 s to reward or no reward): {100 * excl.mean():.2f}% per session (paper 2.71% + 0.142%)")

**Result: Not reproduced.** The mean time from the start tone to the reward is 9.30 s (s.e.m. 0.11 s
over 94 sessions; session means 7.39 to 12.66 s), against 7.4 +- 0.1 s. Measured to the
goal it is 9.18 s, and from the end of the 970 ms start tone 8.33 s, so no simple change of
the reference point brings it to 7.4 s. The time per no-sound zone is 1.009 +- 0.020 s under
reading (a) and 0.896 +- 0.016 s under reading (b), both outside 0.86 +- 0.01 s. Only 0.48%
of trials per session take over 20 s, against the paper's 2.71%.

## lick-goal-correlation

> In every session the start-to-first-lick distance correlated with the start-to-goal distance, with r from 0.258 to 0.822 over the 94 sessions (r = 0.798 in the example session).

Source: Results, Licking indicates action-dependent distance estimation, paragraph 2; Figure 1b; Online Methods, Behavioral analysis.

First lick: the first lick onset more than 33 cm from the start (Online Methods, Behavioral
analysis), with the onset read from the lick frequency (header). A variant counts the first
frame of non-zero frequency beyond 33 cm. Readings (a) Pearson and (b) Spearman.

In [ ]:
rows = []
for i, r in enumerate(R):
    for onset in ("change", "any"):
        tr = r["trials"]
        if onset == "change":
            fl = r["first_lick_cm"]
        else:
            fl = r["first_lick_any_cm"]
        v = valid(r) & ~np.isnan(fl)
        pr = stats.pearsonr(fl[v], tr["gd"][v]); sr = stats.spearmanr(fl[v], tr["gd"][v])
        rows.append(dict(session=i, onset=onset, n=v.sum(), r_pearson=pr[0], p_pearson=pr[1], r_spearman=sr[0], p_spearman=sr[1]))
LC = pd.DataFrame(rows)
for onset, t in LC.groupby("onset"):
    for kind in ("pearson", "spearman"):
        rr, pp = t[f"r_{kind}"], t[f"p_{kind}"]
        ok = (rr > 0).all() and (pp < 0.01).all() and rr.min() >= 0.232 and rr.max() <= 0.904
        print(f"lick onset '{onset}', {kind}: r {rr.min():.3f} to {rr.max():.3f}, largest P {pp.max():.3g}, "
              f"sessions with P >= 0.01: {(pp >= 0.01).sum()}, below r 0.232: {(rr < 0.232).sum()}; criterion met: {ok}")
print("lowest five Pearson r (onset 'change'):", np.round(np.sort(LC[LC.onset == "change"].r_pearson)[:5], 3))

**Result: Partially reproduced.** With Pearson's r, every session has a positive correlation with
P < 0.01 (largest P 0.0079), and r spans 0.218 to 0.822; the upper bound matches the
paper's 0.822, but two sessions fall below the criterion's lower bound of 0.232 (0.218 and
0.227). Spearman's coefficient fails more widely (5 sessions with P >= 0.01, r down to
0.151). The lick-onset variant gives the same numbers.

## fig1c-licking-without-sound

> Licking increased as mice approached the goal in all conditions, and in both intermittent conditions it increased even within the no-sound zones (41.9-25.1 and 8.4-0 cm in intermittent 1, 25.1-8.4 cm in intermittent 2).

Source: Figure 1c; Results, Licking indicates action-dependent distance estimation, paragraph 2.

Licks per trial in 4.2 cm bins of goal distance, from the start to reaching the goal,
mean per session and condition; one-sided signed-rank test over the 94 sessions of each
tested bin against the bin 8.4 cm farther from the goal (criterion). Per-mouse medians are
shown.

In [ ]:
LB = np.stack([np.nanmean(r["lick_bins"][valid(r) & (r["trials"]["state"] == c)], axis=0) for r in R for c in range(3)]).reshape(len(R), 3, -1)
edges = F.LICK_EDGES
def bin_of(hi):
    return int(np.argmin(np.abs(edges[:-1] - hi)))
TESTED = [(29.4, 37.8), (21.0, 29.4), (12.6, 21.0), (4.2, 12.6)]   # (upper edge of tested bin, upper edge of bin 8.4 cm farther)
silent = {1: [0, 3], 2: [1, 2]}
res = []
for c in range(3):
    for j, (hi, far) in enumerate(TESTED):
        a, b = LB[:, c, bin_of(hi)], LB[:, c, bin_of(far)]
        p, z, n = F.wilcoxon_z(a, b, alternative="greater")
        res.append(dict(condition=COND_NAMES[c], tested=f"{hi:.1f}-{hi - 4.2:.1f}", against=f"{far:.1f}-{far - 4.2:.1f}",
                        median_tested=np.median(a), median_far=np.median(b), z=z, p=p, n=n,
                        no_sound=(c > 0 and j in silent[c])))
FIG1C = pd.DataFrame(res); print(FIG1C.round(4).to_string())
ns = FIG1C[FIG1C.no_sound]; co = FIG1C[FIG1C.condition == "continuous"]
print("no-sound comparisons with P < 0.01:", int((ns.p < 0.01).sum()), "of", len(ns), "; continuous:", int((co.p < 0.01).sum()), "of 4")
for subj, ix in sessions.groupby("subject").groups.items():
    d = [np.median(LB[ix, c, bin_of(hi)] - LB[ix, c, bin_of(far)]) for c in (1, 2) for j, (hi, far) in enumerate(TESTED) if j in silent[c]]
    print(f"  {subj}: median rise in the four no-sound comparisons", np.round(d, 3))

**Result: Reproduced.** Licking rises with P < 0.01 in all four no-sound comparisons (intermittent 1,
29.4-25.2 and 4.2-0 cm; intermittent 2, 21.0-16.8 and 12.6-8.4 cm; z 5.41 to 7.37) and in
all four continuous comparisons (z 7.42 to 8.42). The bins are at multiples of 4.2 cm
(the claims file writes 29.3-25.1). Five of eight mice show a rise in all four no-sound
comparisons; mice 1, 3 and 6 each have one comparison without a rise.

## supp1-licking-speed

> During no-sound zones licking rose more steeply in fast-locomotion trials than in slow ones, in both intermittent conditions.

Source: Supplementary Figure 1; Results, Licking indicates action-dependent distance estimation, paragraph 2.

Slope of lick frequency on time within a no-sound zone, per trial; trials of each session
split into halves by speed; mean slope of each half per session; signed-rank test over
sessions. O9 names two readings of the zone. The panel titles of Supplementary Fig. 1 give
it: "Intermittent 1: 8.4 cm to 0 cm from goal" and "Intermittent 2: 25 cm to 8.4 cm from
goal", the no-sound zone nearest the goal, reading (b). Both speed readings are run.

In [ ]:
def supp1(speed_reading, zone_reading):
    out = []
    for c in (1, 2):
        fast_s, slow_s = [], []
        for r in R:
            tr = r["trials"]; m = valid(r) & (tr["state"] == c)
            sl = r["lick_slope"][m]; spz = r["lick_slope_speed"][m]
            if zone_reading == "a":
                slope = np.nanmean(sl, axis=1); zsp = np.nanmean(spz, axis=1)
            else:
                zi = 3 if c == 1 else 2
                slope = sl[:, zi]; zsp = spz[:, zi]
            spd = r["trial_speed"][m] if speed_reading == "a" else zsp
            ok = ~np.isnan(slope) & ~np.isnan(spd)
            slope, spd = slope[ok], spd[ok]
            if len(slope) < 2:
                fast_s.append(np.nan); slow_s.append(np.nan); continue
            med = np.median(spd)
            order = np.argsort(spd); h = len(spd) // 2
            slow_s.append(slope[order[:h]].mean()); fast_s.append(slope[order[-h:]].mean())
        p, z, n = F.wilcoxon_z(np.array(fast_s), np.array(slow_s))
        out.append(dict(condition=COND_NAMES[c], speed=speed_reading, zones=zone_reading, median_fast=np.nanmedian(fast_s),
                        median_slow=np.nanmedian(slow_s), z=z, p=p, n=n))
    return out
S1 = pd.DataFrame([x for s in "ab" for zr in "ab" for x in supp1(s, zr)])
print(S1.round(5).to_string())
print("readings with fast > slow and P < 0.01 in both conditions:",
      int(S1.groupby(["speed", "zones"]).apply(lambda t: ((t.z > 0) & (t.p < 0.01)).all()).sum()), "of 4")

**Result: Reproduced.** In the zone the figure uses (reading b), fast trials have steeper licking
slopes than slow trials in both conditions under both speed readings: speed over the trial,
z = 3.06 (P = 0.0022) and 2.72 (P = 0.0066); speed in the zone, z = 3.21 (P = 0.0013) and
2.92 (P = 0.0035). Averaging over every no-sound zone (reading a), the difference is not
significant (P 0.013 to 0.79); that reading is shown only, since the figure fixes the zone.

Per-session counts and the PPC against PM test used by the next claims.

In [ ]:
def per_session(c, col):
    s = pd.Series(c[col].values).groupby(NT.session.values).agg(["sum", "mean"])
    return s.reindex(range(len(R)))

def ppc_vs_pm(vals, alternative="two-sided"):
    out = []
    for L in (2, 3, 5):
        a = vals[(sessions.area == "PPC") & (sessions.layer == L)]; b = vals[(sessions.area == "PM") & (sessions.layer == L)]
        p, z = F.mwu(a, b, alternative)
        out.append(dict(layer=L, ppc=np.median(a), pm=np.median(b), z=z, p=p, n=f"{len(a)}, {len(b)}"))
    return pd.DataFrame(out)

## neuron-classes

> Goal-distance neurons numbered 1,116, 1,054 and 483 in PPC layers 2, 3 and 5 and 601, 479 and 129 in PM; of these, 744, 854 and 367 in PPC and 193, 204 and 54 in PM were after-reaching neurons; per session, on average 65.6, 70.3 and 34.5 in PPC and 37.6, 31.9 and 7.59 in PM were goal-distance neurons.

Source: Figure 2d; Figure 3c legend; Supplementary Figure 6 legend; Results, Distance prediction with dynamic Bayesian inference, paragraph 2.

The classification of the claims file (Online Methods, Task-relevant neurons), under O1
(baseline zero or the inter-trial mean) and O3 (k-means on the normalized 16-frame pre-goal
trace, or on its mean); O2 does not change these classes. The k-means is run once on the
after-reaching group pooled over all sessions, with a fixed seed and 20 starts. The paper
does not say whether it clustered per session or pooled; pooling is a standard choice.

In [ ]:
EXP_GD = [1116, 1054, 483, 601, 479, 129]; EXP_AFTER = [744, 854, 367, 193, 204, 54]
rows = []
for rd, c in CLS_GD.items():
    g = NT.group
    gd = [int(c.gd[g == x].sum()) for x in GROUPS]; af = [int(c.after[g == x].sum()) for x in GROUPS]
    within = sum(abs(a - e) <= 0.1 * e for a, e in zip(gd + af, EXP_GD + EXP_AFTER))
    rows.append(dict(reading=f"O1 {rd[0]}, O3 {rd[2]}", **{f"gd {x}": v for x, v in zip(GROUPS, gd)},
                     **{f"after {x}": v for x, v in zip(GROUPS, af)}, within_10pct=within))
    print(f"O1 {rd[0]} O3 {rd[2]}: before", [int(c.before[g == x].sum()) for x in GROUPS], "reward", [int(c.reward[g == x].sum()) for x in GROUPS],
          "lick", [int(c.lick[g == x].sum()) for x in GROUPS])
NC = pd.DataFrame(rows).set_index("reading")
print(NC.T.to_string())
print("expected gd", EXP_GD, "after", EXP_AFTER)
for rd, c in CLS_GD.items():
    per = [c.gd[NT.group == x].sum() / (sessions.group == x).sum() for x in GROUPS]
    print(f"O1 {rd[0]} O3 {rd[2]}: goal-distance neurons per session", np.round(per, 1), "(paper 65.6, 70.3, 34.5, 37.6, 31.9, 7.59)")
c0 = CLS_GD[("a", "67", "a")]
print("by mouse, reading O1 a O3 a, goal-distance neurons and share of ROIs:")
for subj in sorted(sessions.subject.unique()):
    m = NT.session.isin(sessions.i[sessions.subject == subj])
    print(f"  {subj}: {int(c0.gd[m].sum())} of {int(m.sum())} ({100 * c0.gd[m].mean():.1f}%)")

**Result: Not reproduced.** Under the zero baseline (O1 a) 2 of the 12 counts are within 10% (PM
layers 2 and 3 goal-distance neurons, 659 and 512); the others are 12 to 41% higher than the
paper's (PPC goal-distance 1,353, 1,292 and 555 against 1,116, 1,054 and 483; after-reaching
942, 1,061, 431 against 744, 854, 367). Under the inter-trial baseline (O1 b) none is within
10%. O3 changes no count by more than 3. The pattern is the paper's: PPC has more
goal-distance and after-reaching neurons than PM, and layer 5 the fewest.

## fig2d-task-relevant-proportions

> Task-relevant neurons made up 41.3, 33.8 and 20.0% of active neurons in PPC layers 2, 3 and 5 and 19.6, 16.7 and 5.70% in PM, more in PPC than in PM in every layer.

Source: Results, Goal-distance representation in PPC and PM, paragraph 4; Figure 2d.

In [ ]:
EXP_TR = [41.3, 33.8, 20.0, 19.6, 16.7, 5.70]
for rd in READINGS:
    if rd[2] != "a":
        continue                        # O3 does not change task relevance
    c = CLS[rd]
    ps = per_session(c, "task")
    mean_ps = [100 * ps["mean"][sessions.group == x].mean() for x in GROUPS]
    pooled = [100 * c.task[NT.group == x].mean() for x in GROUPS]
    print(f"O1 {rd[0]}, O2 0-{rd[1]} cm: (a) mean of session %", np.round(mean_ps, 1),
          "within:", sum(abs(a - e) <= 0.1 * e for a, e in zip(mean_ps, EXP_TR)),
          "| (b) pooled %", np.round(pooled, 1), "within:", sum(abs(a - e) <= 0.1 * e for a, e in zip(pooled, EXP_TR)))
    print(ppc_vs_pm(100 * ps["mean"]).round(5).to_string(index=False))
print("expected", EXP_TR)

**Result: Not reproduced.** No percentage is within tolerance under any reading. Under O1 a the means of
per-session percentages are 47.6, 38.5 and 24.0% (PPC) and 27.3, 23.4 and 8.5% (PM) against
41.3, 33.8, 20.0, 19.6, 16.7 and 5.70%. PPC exceeds PM in every layer; the Mann-Whitney test
gives P < 0.05 in layers 2 and 5 under every reading, and in layer 3 P = 0.051 under O1 a and
0.023 to 0.025 under O1 b.

## fig2d-goal-distance-ppc-more

> PPC had more goal-distance neurons than PM in all three layers.

Source: Results, Goal-distance representation in PPC and PM, paragraph 4; Figure 2d.

In [ ]:
for rd, c in CLS_GD.items():
    ps = per_session(c, "gd")
    for meas, vals in (("proportion", ps["mean"]), ("count", ps["sum"])):
        t = ppc_vs_pm(vals)
        print(f"O1 {rd[0]} O3 {rd[2]}, {meas}: PPC higher with P < 0.05 in", int(((t.z > 0) & (t.p < 0.05)).sum()), "of 3 layers")
        print(t.round(5).to_string(index=False))

**Result: Reproduced.** PPC has more goal-distance neurons than PM in all three layers, as counts and as
proportions of active neurons, under all four readings of O1 and O3 (P 0.0002 to 0.013).

## fig2d-after-vs-before

> PPC had more after-reaching than before-reaching neurons in every layer, whereas PM tended the other way, significantly in layer 2.

Source: Results, Goal-distance representation in PPC and PM, paragraph 4; Figure 2d.

The readings disagree (below). The zero baseline (O1 a) is adopted as an assumption for all
claims that use the classification: dR/R is defined against the session mean of R, so zero is
its own reference, and under it 91 of 94 sessions have goal-distance neurons, the paper's
count ("sessions with 0 goal-distance neurons (3 sessions of 94) were excluded"), where the
inter-trial baseline gives 94 of 94 (decoder cell, below).

In [ ]:
for rd, c in CLS_GD.items():
    pa, pb = per_session(c, "after"), per_session(c, "before")
    for meas in ("sum", "mean"):
        out = []
        for x in GROUPS:
            m = (sessions.group == x).values
            p, z, n = F.wilcoxon_z(pa[meas][m].values, pb[meas][m].values)
            out.append(dict(group=x, after=np.median(pa[meas][m]), before=np.median(pb[meas][m]), z=z, p=p, n=int(m.sum())))
        print(f"O1 {rd[0]} O3 {rd[2]}, {'counts' if meas == 'sum' else 'proportions'} (z > 0: after-reaching more)")
        print(pd.DataFrame(out).round(5).to_string(index=False))

**Result: Reproduced, under the assumption O1 a.** With the zero baseline, PPC has more after-reaching
than before-reaching neurons in every layer (counts P = 0.0086, 0.0038 and 0.011;
proportions 0.0086, 0.0038, 0.0060), PM layer 2 has more before-reaching neurons
(P = 0.0038), and PM layers 3 and 5 do not differ (P 0.39 to 0.64). With the inter-trial
baseline, PPC does not differ in any layer (P 0.07 to 0.52) and PM has more before-reaching
neurons in all three layers.

## fig2d-layer-distance-preference

> In both PPC and PM, goal-distance neurons in layers 3 and 5 preferred distances closer to the goal than those in layer 2.

Source: Figure 2d insets and legend; Results, Goal-distance representation in PPC and PM, paragraph 4.

In [ ]:
for rd, c in CLS_GD.items():
    out = []
    for area in ("PPC", "PM"):
        pref = {L: NT.dist_pref[c.gd.values & (NT.area == area).values & (NT.layer == L).values].values for L in (2, 3, 5)}
        for a, b in ((2, 3), (2, 5), (3, 5)):
            p, z = F.mwu(pref[a], pref[b])
            out.append(dict(area=area, pair=f"L{a} vs L{b}", median_a=np.median(pref[a]), median_b=np.median(pref[b]), z=z, p=p,
                            n=f"{len(pref[a])}, {len(pref[b])}"))
    print(f"O1 {rd[0]} O3 {rd[2]} (z > 0: first layer prefers farther distances)")
    print(pd.DataFrame(out).round(5).to_string(index=False))

**Result: Reproduced, under the assumption O1 a.** With the zero baseline, layers 3 and 5 prefer
distances nearer the goal than layer 2 in both areas with P < 0.01 (PPC z = 7.93 and 4.59;
PM z = 4.19, P = 2e-5, and 2.61, P = 0.0083). With the inter-trial baseline, PM layer 5 does
not differ from layer 2 (P = 0.38, layer 5 median farther from the goal).

Slopes of condition-averaged activity of each after-reaching neuron against approach, in the
zones 25.1-8.4 and 8.4-0 cm (equation 5), under the two trace readings of the claims file.

In [ ]:
from scipy.ndimage import uniform_filter1d
ZB = {"25.1-8.4": (F.DCENT > 8.4) & (F.DCENT <= 25.1), "8.4-0": F.DCENT <= 8.4}
def slopes_all(trace_reading):
    """Slope of each neuron's condition-averaged activity against approach (rise per cm toward the
    goal, equation 5), per condition and zone. Reading (a) the 0.42 cm bins, (b) after the 8.37 cm
    moving average of the classification."""
    y = COND if trace_reading == "a" else uniform_filter1d(np.nan_to_num(COND), F.SMOOTH_D, axis=2, mode="nearest")
    out = {}
    for z, m in ZB.items():
        x = F.DCENT[m]; yy = y[:, :, m]
        xc = x - x.mean()
        out[z] = -((yy - np.nanmean(yy, axis=2, keepdims=True)) * xc).sum(2) / (xc ** 2).sum()   # neurons x conditions
    return out
SLOPES = {tr: slopes_all(tr) for tr in "ab"}
SOUND_COND = {"25.1-8.4": 1, "8.4-0": 2}          # condition with sound in the zone
SILENT_COND = {"25.1-8.4": 2, "8.4-0": 1}

def slope_tests(rd, tr, conds, areas=("PPC", "PM")):
    c = CLS_GD[rd]; out = []
    for area in areas:
        for L in (2, 3, 5):
            m = c.after.values & (NT.area == area).values & (NT.layer == L).values
            for cond in conds:
                for z in ZB:
                    s = SLOPES[tr][z][m, cond]
                    p, zz, n = F.wilcoxon_z(s)
                    out.append(dict(area=area, layer=L, condition=COND_NAMES[cond], zone=z, sound=(cond == 0 or SOUND_COND[z] == cond),
                                    median=np.nanmedian(s), z=zz, p=p, n=n))
    return pd.DataFrame(out)

## fig3c-continuous-ramp

> In the continuous condition, after-reaching neurons increased their activity as mice approached the goal, in both PPC and PM.

Source: Figure 3c; Results, PPC represents goal distance estimated from action, paragraph 1.

In [ ]:
for rd in CLS_GD:
    for tr in "ab":
        t = slope_tests(rd, tr, [0])
        ok = ((t["median"] > 0) & (t.p < 0.01)).sum()
        print(f"O1 {rd[0]} O3 {rd[2]}, trace reading {tr}: positive with P < 0.01 in {ok} of {len(t)}")
        if rd == ("a", "67", "a"):
            print(t.round(5).to_string(index=False))

**Result: Reproduced.** In the continuous condition the median slope is positive with P < 0.01 in all
12 area, layer and zone tests, under all four classification readings and both trace readings
(z 6.3 to 27.9 in the reading shown).

## fig3c-ppc-ramp-without-sound

> In the intermittent conditions, PPC after-reaching neurons in all layers increased activity toward the goal both with and without sound.

Source: Figure 3c; Supplementary Figure 8a; Results, PPC represents goal distance estimated from action, paragraph 1.

In [ ]:
for rd in CLS_GD:
    for tr in "ab":
        t = slope_tests(rd, tr, [1, 2], areas=("PPC",))
        ok = ((t["median"] > 0) & (t.p < 0.01)).sum()
        print(f"O1 {rd[0]} O3 {rd[2]}, trace reading {tr}: positive with P < 0.01 in {ok} of {len(t)}; "
              f"no-sound tests {((t['median'] > 0) & (t.p < 0.01) & ~t.sound).sum()} of {(~t.sound).sum()}")
        if rd == ("a", "67", "a"):
            print(t.round(6).to_string(index=False))

**Result: Reproduced.** In both intermittent conditions PPC after-reaching neurons in all layers have
positive median slopes with P < 0.01 in all 12 tests, including the 6 no-sound tests (z 5.6
to 15.4 without sound in the reading shown), under every reading.

## supp8a-pm-no-sound

> In PM, after-reaching neurons showed positive slopes in some no-sound zones, but in the intermittent 1 condition their slopes without sound were not significantly positive.

Source: Supplementary Figure 8a; Results, PPC represents goal distance estimated from action, paragraph 1.

The trace reading is taken from the Methods, which define y(distance) as "the neural activity
trace dR/R at a given goal distance binned every 0.42 cm" with no moving average: reading
(a). Reading (b) and O1 b are shown.

In [ ]:
for rd in CLS_GD:
    for tr in "ab":
        t = slope_tests(rd, tr, [1, 2], areas=("PM",))
        t = t[~t.sound]
        i2 = t[t.condition == "intermittent 2"]; i1 = t[t.condition == "intermittent 1"]
        a = ((i2["median"] > 0) & (i2.p < 0.01)).sum(); b = ((i1["median"] > 0) & (i1.p < 0.01)).sum()
        print(f"O1 {rd[0]} O3 {rd[2]}, trace {tr}: intermittent 2 (25.1-8.4) positive P < 0.01 in {a} of 3; intermittent 1 (8.4-0) in {b} of 3")
        print("   ", t[["layer", "condition", "zone", "median", "z", "p", "n"]].round(5).to_string(index=False).replace("\n", "\n    "))

**Result: Reproduced, under the assumptions O1 a and trace reading (a).** PM after-reaching neurons
have positive slopes with P < 0.01 in intermittent 2 at 25.1-8.4 cm in all three layers, and
in no layer in intermittent 1 at 8.4-0 cm (P = 0.47, 0.57 and 0.017). With the moving average
(b), or the inter-trial baseline, layer 5 in intermittent 1 reaches P < 0.01 (0.0026 and
0.006), so the claim's absence part fails there.

## supp8b-slopes-lower-without-sound

> Slopes of after-reaching activity were lower without sound than with sound.

Source: Supplementary Figure 8b.

In [ ]:
for rd in CLS_GD:
    c = CLS_GD[rd]
    for tr in "ab":
        out = []
        for area in ("PPC", "PM"):
            for L in (2, 3, 5):
                m = c.after.values & (NT.area == area).values & (NT.layer == L).values
                for z in ZB:
                    w, wo = SLOPES[tr][z][m, SOUND_COND[z]], SLOPES[tr][z][m, SILENT_COND[z]]
                    p, zz, n = F.wilcoxon_z(wo, w)
                    out.append(dict(area=area, layer=L, zone=z, with_sound=np.nanmedian(w), without=np.nanmedian(wo), z=zz, p=p, n=n))
        t = pd.DataFrame(out)
        low = (t.z < 0) & (t.p < 0.01)
        print(f"O1 {rd[0]} O3 {rd[2]}, trace {tr}: without lower P < 0.01 in PPC {low[t.area == 'PPC'].sum()} of 6, PM {low[t.area == 'PM'].sum()} of 6")
        if rd == ("a", "67", "a"):
            print(t.round(5).to_string(index=False))

**Result: Reproduced, under the assumption O1 a.** Slopes without sound are lower than with sound with
P < 0.01 in all six PPC comparisons and in four of six PM comparisons (PM layer 5, P = 0.029
and 0.017), under both trace readings. With the inter-trial baseline PM has three of six.

## supp8c-ppc-steeper-than-pm

> Without sound, PPC after-reaching neurons in layers 2 and 3 had steeper slopes than PM, while slopes in PM were equal to or lower than in PPC.

Source: Supplementary Figure 8c; Results, PPC represents goal distance estimated from action, paragraph 1.

In [ ]:
for rd in CLS_GD:
    c = CLS_GD[rd]
    for tr in "ab":
        out = []
        for L in (2, 3, 5):
            for z in ZB:
                sel = lambda area: c.after.values & (NT.area == area).values & (NT.layer == L).values
                a, b = SLOPES[tr][z][sel("PPC"), SILENT_COND[z]], SLOPES[tr][z][sel("PM"), SILENT_COND[z]]
                p, zz = F.mwu(a, b)
                out.append(dict(layer=L, zone=z, ppc=np.nanmedian(a), pm=np.nanmedian(b), z=zz, p=p, n=f"{len(a)}, {len(b)}"))
        t = pd.DataFrame(out)
        hi = (t.z > 0) & (t.p < 0.05)
        ok = hi[t.layer == 3].all() and hi[t.layer == 2].any() and not ((t.z < 0) & (t.p < 0.05)).any()
        print(f"O1 {rd[0]} O3 {rd[2]}, trace {tr}: criterion met {ok}")
        print("   ", t.round(5).to_string(index=False).replace("\n", "\n    "))

**Result: Partially reproduced, under the assumption O1 a.** Without sound, PPC slopes exceed PM in
layers 2 and 3 at 8.4-0 cm (P = 0.0010 and 0.0006 with trace reading a), and PM is steeper in
no comparison, but layer 3 at 25.1-8.4 cm does not differ (P = 0.59 and 0.10 under the two
trace readings). With the inter-trial baseline and the moving average all parts hold.

## supp9-preserved-representation

> PPC neurons kept the distance preference they had in the continuous condition in the intermittent conditions, especially near the goal, more than PM neurons did.

Source: Supplementary Figure 9b; Results, PPC represents goal distance estimated from action, paragraph 2.

The before-reaching groups are not given (claims file). The equal quarters of reading (a)
give group sizes close to the n of Supplementary Fig. 9a and the zone borders of reading (b)
do not (group sizes printed below), so (a) is adopted as an assumption. The claims file
reads the legend's "difference" as absolute. The y axis of Supplementary Fig. 9b, "Difference
to continuous condition (cm)", runs from 37.7 to -58.6, which suggests a signed difference;
the claim is tested as the file has it, signed medians are shown, and a reader is asked.

In [ ]:
def supp9(rd, groups):
    c = CLS_GD[rd]; out = []
    pc = COND_PREF[:, 0]
    for ci in (1, 2):
        diff = np.abs(pc - COND_PREF[:, ci])
        for L in (2, 3, 5):
            for gname in list(groups) + ["after-reaching"]:
                if gname == "after-reaching":
                    gm = c.after.values
                else:
                    hi, lo = groups[gname]
                    gm = c.before.values & (pc <= hi) & (pc > lo)
                sel = lambda area: gm & (NT.area == area).values & (NT.layer == L).values
                a, b = diff[sel("PPC")], diff[sel("PM")]
                p, z = F.mwu(a, b)
                out.append(dict(condition=COND_NAMES[ci], layer=L, group=gname, ppc=np.nanmedian(a), pm=np.nanmedian(b), z=z, p=p,
                                n=f"{np.sum(~np.isnan(a))}, {np.sum(~np.isnan(b))}"))
    return pd.DataFrame(out)
G9 = {"a": {"67-50.25": (67.2, 50.25), "50.25-33.5": (50.25, 33.5), "33.5-16.75": (33.5, 16.75), "16.75-0": (16.75, 0)},
      "b": {"67-58.6": (67.2, 58.6), "58.6-25.1": (58.6, 25.1), "25.1-8.4": (25.1, 8.4), "8.4-0": (8.4, 0)}}
for rd in CLS_GD:
    for gr in "ab":
        t = supp9(rd, G9[gr])
        af = t[(t.group == "after-reaching") & (t.condition == "intermittent 1")]
        a = ((af.z < 0) & (af.p < 0.05)).sum(); pm_better = ((t.z > 0) & (t.p < 0.05)).sum()
        print(f"O1 {rd[0]} O3 {rd[2]}, groups {gr}: intermittent 1 after-reaching PPC smaller with P < 0.05 in {a} of 3 layers; "
              f"PM significantly smaller in {pm_better} of {len(t)}")
        if rd == ("a", "67", "a"):
            print(t.round(5).to_string(index=False))
# group sizes against Supplementary Fig. 9a (before-reaching groups; paper n per layer)
PAPER9 = {"PPC L2": [82, 53, 66, 171], "PPC L3": [14, 19, 22, 145], "PPC L5": [26, 8, 16, 66],
          "PM L2": [60, 38, 87, 223], "PM L3": [23, 16, 76, 160], "PM L5": [17, 4, 18, 36]}
c = CLS_GD[("a", "67", "a")]; pc = COND_PREF[:, 0]
for gr in "ab":
    err = 0
    for g in GROUPS:
        m = c.before.values & (NT.group == g).values
        n = [int((m & (pc <= hi) & (pc > lo)).sum()) for hi, lo in G9[gr].values()]
        frac = np.array(n) / max(sum(n), 1); pf = np.array(PAPER9[g]) / sum(PAPER9[g])
        err += np.abs(frac - pf).sum()
        print(f"groups {gr}, {g}: n {n} (paper {PAPER9[g]})")
    print(f"groups {gr}: summed absolute difference of group shares from the paper's, {err:.2f}")
# signed differences, shown only
c = CLS_GD[("a", "67", "a")]
for area in ("PPC", "PM"):
    m = c.after.values & (NT.area == area).values
    print(f"signed difference continuous minus intermittent 1, after-reaching {area}: median {np.nanmedian(COND_PREF[m, 0] - COND_PREF[m, 1]):.2f} cm")

**Result: Reproduced, under the assumptions O1 a and groups (a).** In intermittent 1, PPC
after-reaching neurons have a smaller absolute change of distance preference than PM in all
three layers (P < 0.05), and PM is significantly smaller in none of the 30 comparisons. With
zone-border groups (b) one comparison of 30 favours PM; with the inter-trial baseline PPC is
smaller in two of three layers.

## Decoder

The probabilistic decoder of the Methods for every session with goal-distance neurons, under
the four classification readings and the two shifts of O7, with a Gaussian-likelihood decoder
and a decoder trained on permuted distances for comparison. Each session has its own seeded
generator for the permutation.

In [ ]:
import time as _time
from concurrent.futures import ThreadPoolExecutor
DEC_CONFIGS = [(rd, sh) for rd in CLS_GD for sh in (0.01, 0.1)]
roi_index = {(s, r): i for i, (s, r) in enumerate(zip(NT.session, NT.roi))}

def decode_session(i):
    r = R[i]; d = r["dec"]; out = {}
    rng = np.random.default_rng(1000 + i)
    for rd in CLS_GD:
        gd = CLS_GD[rd].gd.values
        keep = np.array([gd[roi_index[(i, n)]] for n in d["neurons"]], bool) if len(d["neurons"]) else np.zeros(0, bool)
        if keep.sum() == 0:
            continue
        dd = dict(d, n=d["n"][:, keep], min_all=d["min_all"][keep])
        for sh in (0.01, 0.1):
            out[(rd, sh, "lognormal")] = F.decode(dd, sh)
        out[(rd, 0.01, "permuted")] = F.decode(dd, 0.01, permute=True, rng=rng)
        out[(rd, None, "gaussian")] = F.decode(dd, likelihood="gaussian")
        out[(rd, "n")] = int(keep.sum())
    return out

t0 = _time.time()
with ThreadPoolExecutor(4) as ex:
    DEC = list(ex.map(decode_session, range(len(R))))
print(f"decoding took {_time.time() - t0:.0f} s")
FR = pd.concat([pd.DataFrame(dict(session=i, area=R[i]["meta"]["area"], layer=R[i]["meta"]["layer"], trial=R[i]["dec"]["trial"],
                                  state=R[i]["dec"]["state"], x=R[i]["dec"]["x"], time=R[i]["dec"]["time"])) for i in range(len(R))],
               ignore_index=True)
FR["group"] = FR.area + " L" + FR.layer.astype(str)
FR["flat"] = F.flat_rmse(FR.x.values)
def col(key, k):
    """Concatenate one decoder output (0 RMSE, 1 MAP, 2 uncertainty) over sessions; NaN where a
    session has no goal-distance neuron under the reading."""
    return np.concatenate([DEC[i][key][k] if key in DEC[i] else np.full(len(R[i]["dec"]["x"]), np.nan) for i in range(len(R))])
for rd in CLS_GD:
    n_ok = sum((rd, "n") in DEC[i] for i in range(len(R)))
    print(f"O1 {rd[0]} O3 {rd[2]}: sessions decoded {n_ok} of {len(R)}; neurons per session median "
          f"{np.median([DEC[i].get((rd, 'n'), 0) for i in range(len(R))]):.0f}")
MAIN = (("a", "67", "a"), 0.01)
print("frames", len(FR), "range of x", FR.x.min().round(2), FR.x.max().round(2))
print(f"median RMSE, main reading: continuous {np.nanmedian(col((*MAIN, 'lognormal'), 0)[FR.state == 0]):.2f} cm, "
      f"flat prior {FR.flat.median():.2f} cm")

## fig4-above-chance

> Decoders built from goal-distance neurons estimated goal distance above chance in both PPC and PM.

Source: Results, Distance prediction with dynamic Bayesian inference, paragraph 2; Figure 4b.

Chance reading (a), the flat prior's RMSE at each frame, is adopted as an assumption: the
decoder uses a flat prior, and reading (b), distances permuted across trials, keeps the
fall of distance through every trial, so its decoder still decodes distance (median RMSE
within 1 cm of the real decoder in the main reading).

In [ ]:
def label(rd, sh):
    return f"O1 {rd[0]} O3 {rd[2]}, shift {sh}"
for rd, sh in DEC_CONFIGS:
    rm = col((rd, sh, "lognormal"), 0)
    for chance in ("flat", "permuted"):
        ch = FR.flat.values if chance == "flat" else col((rd, 0.01, "permuted"), 0)
        out = []
        for gname in GROUPS:
            for c in range(3):
                m = ((FR.group == gname) & (FR.state == c)).values & ~np.isnan(rm)
                p, z, n = F.wilcoxon_z(rm[m], ch[m])
                out.append(dict(group=gname, condition=COND_NAMES[c], rmse=np.median(rm[m]), chance=np.median(ch[m]), z=z, p=p, n=n))
        t = pd.DataFrame(out)
        print(f"{label(rd, sh)}, chance {chance}: decoder lower with P < 0.01 in {((t.z < 0) & (t.p < 0.01)).sum()} of 18; "
              f"z {t.z.min():.1f} to {t.z.max():.1f}")
        if (rd, sh) == MAIN:
            print(t.round(4).to_string(index=False))

**Result: Reproduced, under the assumption of chance reading (a).** Decoder RMSE is below the flat
prior's with P < 0.01 in all 18 settings under every classification and shift reading (z -88
to -366; median RMSE 11.1 to 17.0 cm in PPC against about 27 cm). Against permuted-distance
decoders, 17 of 18 settings are lower in six of eight readings and 18 of 18 in two; in the
main reading the exception is PPC layer 3 in intermittent 1 (z = +13.6, worse than permuted).

Per-trial means in goal-distance bins, PPC against PM, used by the next three claims.

In [ ]:
def trial_bins(v, width):
    """Per trial mean of a per-frame value v in goal-distance bins of the given width (cm)."""
    b = np.floor(FR.x.values / width).astype(int)
    df = pd.DataFrame(dict(session=FR.session, trial=FR.trial, state=FR.state, group=FR.group, b=b, v=v, x=FR.x))
    return df.groupby(["session", "trial", "state", "group", "b"], as_index=False)[["v", "x"]].mean()

def ppc_pm_bins(v, width, layer=2, conds=(0, 1, 2), err=None):
    tb = trial_bins(v, width)
    tb = tb[tb.group.isin([f"PPC L{layer}", f"PM L{layer}"]) & tb.state.isin(conds)]
    grp = {k: g.v.values for k, g in tb.groupby(["state", "b", "group"])}
    out = []
    for c in conds:
        for b in sorted(tb.b.unique()):
            a = grp.get((c, b, f"PPC L{layer}"), np.zeros(0)); m = grp.get((c, b, f"PM L{layer}"), np.zeros(0))
            p, z = F.mwu(a, m)
            out.append(dict(condition=COND_NAMES[c], bin=f"{(b + 1) * width:.2f}-{b * width:.2f}", lo=b * width,
                            ppc=np.median(a) if len(a) else np.nan, pm=np.median(m) if len(m) else np.nan, z=z, p=p, n=f"{len(a)}, {len(m)}"))
    return pd.DataFrame(out)

## fig4c-ppc-lower-error

> The decoding error (RMSE) from layer 2 neurons was lower in PPC than in PM.

Source: Figure 4c; Results, Distance prediction with dynamic Bayesian inference, paragraph 2.

In [ ]:
for rd, sh in DEC_CONFIGS:
    for width in (4.2, 0.42):
        t = ppc_pm_bins(col((rd, sh, "lognormal"), 0), width)
        line = []
        ok = True
        for c in COND_NAMES:
            tc = t[t.condition == c]
            lower = (tc.ppc < tc.pm).mean(); sig = ((tc.z < 0) & (tc.p < 0.01)).sum(); pm_sig = ((tc.z > 0) & (tc.p < 0.01)).sum()
            ok &= lower > 0.5 and sig >= 1 and pm_sig == 0
            line.append(f"{c}: PPC lower in {100 * lower:.0f}% of {len(tc)} bins, sig {sig}, PM sig {pm_sig}")
        print(f"{label(rd, sh)}, bins {width} cm: criterion met {ok}; " + "; ".join(line))
        if (rd, sh) == MAIN and width == 4.2:
            print(t.round(5).to_string(index=False))
for L in (3, 5):
    t = ppc_pm_bins(col((*MAIN, "lognormal"), 0), 4.2, layer=L)
    print(f"layer {L}, main reading: PPC lower in {(t.ppc < t.pm).mean() * 100:.0f}% of bins, significant (P < 0.01) {((t.z < 0) & (t.p < 0.01)).sum()}, PM significant {((t.z > 0) & (t.p < 0.01)).sum()} of {len(t)}")

**Result: Reproduced, under the assumption O1 a.** In layer 2 and the main reading, PPC has the lower
median RMSE in 100%, 88% and 88% of the 4.2 cm bins of the three conditions, significantly
lower (P < 0.01) in 12, 8 and 8 bins, and PM is significantly lower in none. Under the zero
baseline the criterion is met with both shifts and both bin widths.
Under the inter-trial baseline with shift 0.1 and 0.42 cm bins one bin favours PM.

## fig4d-map-near-goal

> PPC and PM gave similar MAP estimates, but near the goal PPC estimates were better than PM.

Source: Figure 4d; Results, Distance prediction with dynamic Bayesian inference, paragraph 2.

The bin width of O8 is open. The 4.2 cm bins, the decoder's bin width and the tick spacing
of Fig. 4c-e, are adopted as an assumption; 0.42 cm bins are shown.

In [ ]:
for rd, sh in DEC_CONFIGS:
    for width in (4.2, 0.42):
        mp = col((rd, sh, "lognormal"), 1)
        t = ppc_pm_bins(mp, width, conds=(0,))
        t["centre"] = t.lo + width / 2
        t["ppc_closer"] = np.abs(t.ppc - t.centre) < np.abs(t.pm - t.centre)
        near = t[t.lo < 8.4 - 1e-9]
        a = (near.ppc_closer & (near.p < 0.01)).sum(); b = (~near.ppc_closer & (near.p < 0.01)).sum()
        print(f"{label(rd, sh)}, bins {width} cm: bins in 8.4-0 cm with PPC closer P < 0.01: {a}, PM closer P < 0.01: {b}, of {len(near)}; "
              f"all bins differing P < 0.01: {(t.p < 0.01).sum()} of {len(t)}")
        if (rd, sh) == MAIN and width == 4.2:
            print(t.round(5).to_string(index=False))

**Result: Reproduced, under the assumptions O1 a and 4.2 cm bins.** In the continuous condition both
4.2 cm bins between 8.4 and 0 cm have PPC MAP closer to the actual distance with P < 0.01
(medians 2.64 against 5.53 cm and 7.19 against 10.17 cm), and PM is closer in none; 9 of 16
bins differ overall. With 0.42 cm bins PM is closer with P < 0.01 in 3 to 8 of the 20 bins
near the goal under every reading.

## fig4e-ppc-lower-uncertainty

> The uncertainty (posterior s.d.) of distance estimates was lower in PPC than in PM at every goal distance and in every condition.

Source: Figure 4e; Results, Distance prediction with dynamic Bayesian inference, paragraph 2.

In [ ]:
for rd, sh in DEC_CONFIGS:
    for width in (4.2, 0.42):
        t = ppc_pm_bins(col((rd, sh, "lognormal"), 2), width)
        sig = (t.z < 0) & (t.p < 0.01)
        print(f"{label(rd, sh)}, bins {width} cm: PPC lower with P < 0.01 in {sig.sum()} of {len(t)} bins ({100 * sig.mean():.1f}%)")
        if (rd, sh) == MAIN and width == 4.2:
            print(t.round(5).to_string(index=False))
for L in (3, 5):
    t = ppc_pm_bins(col((*MAIN, "lognormal"), 2), 4.2, layer=L)
    print(f"layer {L}, main reading: PPC lower with P < 0.01 in {((t.z < 0) & (t.p < 0.01)).sum()} of {len(t)}, PM lower in {((t.z > 0) & (t.p < 0.01)).sum()}")

**Result: Reproduced, under the assumption O1 a.** In layer 2 the posterior s.d. is lower in PPC than
PM with P < 0.01 in all 48 bins of 4.2 cm (and all 480 of 0.42 cm) under the zero baseline;
under the inter-trial baseline with shift 0.1, 479 of 480. Layers 3 and 5 are lower in PPC in
all 48 bins.

Per-trial slopes of MAP and uncertainty within each zone (equation 10), decrease per cm of
approach. Frames within 33 cm of the start, where sound always plays, are left out of zones
that are silent in their condition.

In [ ]:
ZONE_OF = np.full(len(FR), -1)
for zi, (hi, lo) in enumerate(F.ZONES):
    ZONE_OF[(FR.x.values <= hi) & (FR.x.values > lo)] = zi
FR["zone"] = ZONE_OF
FR["silent"] = ~F.sound_zone(FR.x.values, np.concatenate([R[i]["dec"]["sdist"] for i in range(len(R))]), FR.state.values)
ZN = ["58.6-41.9", "41.9-25.1", "25.1-8.4", "8.4-0"]

def zone_slopes(v, against="x"):
    """Per trial and zone, slope of v on goal distance (decrease per cm of approach, equation 10),
    or on time (b1 of y = b0 + b1 t) when against='time'. Frames of a zone whose sound state
    differs from the zone's (the first 33 cm from the start) are left out."""
    df = pd.DataFrame(dict(session=FR.session, trial=FR.trial, state=FR.state, group=FR.group, zone=FR.zone, silent=FR.silent,
                           u=FR[against].values, v=v))
    df = df[(df.zone >= 0) & ~np.isnan(df.v)]
    zs = np.array([0, 1, 0, 1]); exp_silent = np.where(df.state == 1, zs[df.zone] == 1, np.where(df.state == 2, zs[df.zone] == 0, False))
    df = df[df.silent.values == exp_silent]
    df["uv"] = df.u * df.v; df["uu"] = df.u ** 2
    g = df.groupby(["session", "trial", "state", "group", "zone"])
    s = g.agg(n=("v", "size"), su=("u", "sum"), sv=("v", "sum"), suv=("uv", "sum"), suu=("uu", "sum")).reset_index()
    s = s[s.n >= 3]
    den = s.suu - s.su ** 2 / s.n
    s["slope"] = (s.suv - s.su * s.sv / s.n) / den.where(den > 1e-9)
    s["silent"] = np.where(s.state == 1, zs[s.zone] == 1, np.where(s.state == 2, zs[s.zone] == 0, False))
    return s

ZS = {}
for rd, sh in DEC_CONFIGS:
    ZS[(rd, sh, "map")] = zone_slopes(col((rd, sh, "lognormal"), 1))
    ZS[(rd, sh, "unc")] = zone_slopes(col((rd, sh, "lognormal"), 2))
print("trial-zone slopes per reading:", len(ZS[(*MAIN, 'map')]))

## fig5a-map-follows-continuous

> In the continuous condition, decoded MAP distances followed the actual distance in all layers of PPC and PM.

Source: Figure 5a; Results, Distance prediction with dynamic Bayesian inference, paragraph 3.

In [ ]:
def against_zero(s, conds):
    out = []
    for gname in GROUPS:
        for c in conds:
            for z in range(4):
                v = s[(s.group == gname) & (s.state == c) & (s.zone == z)].slope
                p, zz, n = F.wilcoxon_z(v.values)
                out.append(dict(group=gname, condition=COND_NAMES[c], zone=ZN[z], silent=bool(c > 0 and ((c == 1) == (z % 2 == 1))),
                                median=np.nanmedian(v), z=zz, p=p, n=n))
    return pd.DataFrame(out)
for rd, sh in DEC_CONFIGS:
    t = against_zero(ZS[(rd, sh, "map")], [0])
    print(f"{label(rd, sh)}: positive with P < 0.01 in {((t['median'] > 0) & (t.p < 0.01)).sum()} of {len(t)}")
    if (rd, sh) == MAIN:
        print(t.round(5).to_string(index=False))

**Result: Reproduced.** In the continuous condition the median MAP slope is positive with P < 0.01 in
all 24 area, layer and zone tests under every reading.

## fig5a-map-decreases-without-sound

> In the intermittent conditions, MAPs decreased with and without sound in most zones, and in every zone in PPC layers 3 and 5.

Source: Figure 5a; Results, Distance prediction with dynamic Bayesian inference, paragraph 3; Discussion, paragraph 3.

In [ ]:
for rd, sh in DEC_CONFIGS:
    t = against_zero(ZS[(rd, sh, "map")], [1, 2])
    pos = (t["median"] > 0) & (t.p < 0.01)
    l35 = t.group.isin(["PPC L3", "PPC L5"])
    print(f"{label(rd, sh)}: PPC L3 and L5 positive P < 0.01 in {pos[l35].sum()} of {l35.sum()}; all groups {pos.sum()} of {len(t)}; "
          f"no-sound {pos[t.silent].sum()} of {t.silent.sum()}")
    if (rd, sh) == MAIN:
        print(t.round(5).to_string(index=False))

**Result: Partially reproduced.** In the intermittent conditions most zones have a positive median MAP
slope with P < 0.01 (39 of 48 tests under O1 a, 42 under O1 b), but PPC layers 3 and 5 reach
it in 13 (O1 a) or 14 (O1 b) of their 16 tests, not all; the failures include the no-sound
zone 8.4-0 cm of intermittent 1, where the median slope is 0.

Sound against no-sound trials, per zone, Mann-Whitney U over trials.

In [ ]:
def sound_vs_silent(s, larger="sound"):
    out = []
    for gname in GROUPS:
        for z, zname in ((2, "25.1-8.4"), (3, "8.4-0")):
            snd = s[(s.group == gname) & (s.zone == z) & (s.state == (1 if z == 2 else 2))].slope
            sil = s[(s.group == gname) & (s.zone == z) & (s.state == (2 if z == 2 else 1))].slope
            p, zz = F.mwu(snd, sil)
            out.append(dict(group=gname, zone=zname, sound=np.nanmedian(snd), no_sound=np.nanmedian(sil), z=zz, p=p,
                            n=f"{snd.notna().sum()}, {sil.notna().sum()}"))
    return pd.DataFrame(out)

## fig5b-map-slope-lower-without-sound

> MAP slopes were shallower without sound than with sound.

Source: Figure 5b; Results, Distance prediction with dynamic Bayesian inference, paragraph 3.

In [ ]:
for rd, sh in DEC_CONFIGS:
    t = sound_vs_silent(ZS[(rd, sh, "map")])
    ok = (t.z > 0) & (t.p < 0.01)
    print(f"{label(rd, sh)}: no-sound lower with P < 0.01 in PPC {ok[t.group.str.startswith('PPC')].sum()} of 6, PM {ok[t.group.str.startswith('PM')].sum()} of 6")
    if (rd, sh) == MAIN:
        print(t.round(5).to_string(index=False))

**Result: Reproduced.** MAP slopes are lower without sound with P < 0.01 in all six PPC comparisons and
in five (O1 a) or four (O1 b) of the six PM comparisons.

## fig5-layer3-steeper-than-layer2

> In intermittent 1, without sound between 8.4 and 0 cm, MAPs decreased more steeply in PPC layer 3 than layer 2, while the s.d. of the estimates did not differ between these layers.

Source: Results, Distance prediction with dynamic Bayesian inference, paragraph 3.

In [ ]:
for rd, sh in DEC_CONFIGS:
    res = []
    for kind in ("map", "unc"):
        s = ZS[(rd, sh, kind)]
        sel = lambda g: s[(s.group == g) & (s.state == 1) & (s.zone == 3)].slope
        p, z = F.mwu(sel("PPC L3"), sel("PPC L2"))
        res.append(f"{kind}: L3 median {np.nanmedian(sel('PPC L3')):.4f}, L2 {np.nanmedian(sel('PPC L2')):.4f}, z {z:.2f}, P {p:.4g}, n {sel('PPC L3').notna().sum()}, {sel('PPC L2').notna().sum()}")
    print(f"{label(rd, sh)}: " + "; ".join(res))

**Result: Not reproduced.** In intermittent 1 at 8.4-0 cm the MAP slope of PPC layer 3 trials is not
steeper than layer 2 (z = 1.48, P = 0.14 in the main reading; P 0.07 to 0.56 over all
readings), so the first part fails. The s.d. slopes do not differ (P 0.06 to 0.66), as the
paper reports. Median MAP slopes in this zone are 0 in both layers.

## fig5cd-uncertainty-updating

> In all PPC layers, between 25.1 and 0 cm, uncertainty of the estimated distance fell more steeply with sound than without, the updating step of dynamic Bayesian inference.

Source: Figure 5c,d; Results, Distance prediction with dynamic Bayesian inference, paragraph 3; Discussion, paragraph 3.

In [ ]:
for rd, sh in DEC_CONFIGS:
    t = sound_vs_silent(ZS[(rd, sh, "unc")])
    ok = (t.z > 0) & (t.p < 0.05)
    print(f"{label(rd, sh)}: decrease larger with sound P < 0.05 in PPC {ok[t.group.str.startswith('PPC')].sum()} of 6, PM {ok[t.group.str.startswith('PM')].sum()} of 6")
    if (rd, sh) == MAIN:
        print(t.round(5).to_string(index=False))

**Result: Reproduced.** Between 25.1 and 0 cm the decrease of uncertainty is larger with sound than
without with P < 0.05 in all six PPC comparisons under every reading (P <= 0.0003 in the main
reading). PM is significant in three (O1 a) or two (O1 b) of six.

## fig5c-uncertainty-decreases-with-sound

> In all PPC layers, the uncertainty of the decoded distance (s.d. of the posterior) decreased during sound presentation between 8.4 and 0 cm.

Source: Figure 5c and legend.

In [ ]:
for rd, sh in DEC_CONFIGS:
    t = against_zero(ZS[(rd, sh, "unc")], [0, 1, 2])
    t = t[t.zone == "8.4-0"]
    key = t.group.str.startswith("PPC") & t.condition.isin(["continuous", "intermittent 2"])
    ok = (t["median"] > 0) & (t.p < 0.01)
    print(f"{label(rd, sh)}: PPC continuous and intermittent 2, decrease P < 0.01 in {ok[key].sum()} of 6; other tests {ok[~key].sum()} of {(~key).sum()}")
    if (rd, sh) == MAIN:
        print(t.round(5).to_string(index=False))

**Result: Reproduced.** At 8.4-0 cm the uncertainty decreases with P < 0.01 in all six PPC tests of the
continuous and intermittent 2 conditions under every reading (z 5.6 to 10.1 in the main
reading).

## supp11-sound-reduces-error

> Decoding errors decreased with sound and increased without it. Within zones, the change per cm of the RMSE difference (intermittent minus continuous) was lower in sound zones than in no-sound zones, in all PPC layers and in PM layers 2 and 3.

Source: Supplementary Figure 11b,c; Results, Distance prediction with dynamic Bayesian inference, paragraph 2.

In [ ]:
def supp11(rmse, zone_reading):
    b = np.floor(FR.x.values / 0.42).astype(int)
    df = pd.DataFrame(dict(session=FR.session, state=FR.state, group=FR.group, b=b, v=rmse)).dropna()
    med = df.groupby(["session", "group", "state", "b"]).v.median().unstack("state")
    out = []
    for c in (1, 2):
        d = (med[c] - med[0]).dropna().reset_index()
        d.columns = ["session", "group", "b", "d"]
        d["x"] = (d.b + 0.5) * 0.42
        for zi, (hi, lo) in enumerate(F.ZONES):
            if zone_reading == "b" and zi < 2:
                continue
            m = (d.x <= hi) & (d.x > lo)
            for (sess, g), dd in d[m].groupby(["session", "group"]):
                if len(dd) >= 3:
                    b1 = -np.polyfit(dd.x, dd.d, 1)[0]
                    silent = (c == 1) == (zi % 2 == 1)
                    out.append(dict(session=sess, group=g, condition=c, zone=zi, silent=silent, b1=b1))
    return pd.DataFrame(out)
for rd, sh in DEC_CONFIGS:
    for zr in "ab":
        t = supp11(col((rd, sh, "lognormal"), 0), zr)
        out = []
        for g in GROUPS:
            a, b = t[(t.group == g) & ~t.silent].b1, t[(t.group == g) & t.silent].b1
            p, z = F.mwu(a, b)
            out.append(dict(group=g, sound=np.median(a), no_sound=np.median(b), z=z, p=p, n=f"{len(a)}, {len(b)}"))
        o = pd.DataFrame(out)
        ok = (o.z < 0) & (o.p < 0.01)
        print(f"{label(rd, sh)}, zones {zr}: b1 lower with sound P < 0.01 in PPC {ok[o.group.str.startswith('PPC')].sum()} of 3, PM {ok[o.group.str.startswith('PM')].sum()} of 3")
        if (rd, sh) == MAIN:
            print(o.round(5).to_string(index=False))

**Result: Reproduced.** The change per cm of the RMSE difference is lower in sound zones than in
no-sound zones with P < 0.01 in all three PPC layers under both readings of n and every
decoder reading. PM reaches it in two layers with four zones and in none or one with two
zones.

## supp13-map-speed

> During no-sound zones, decoded MAPs fell faster in fast-locomotion trials than in slow ones.

Source: Supplementary Figure 13; Results, Distance prediction with dynamic Bayesian inference, paragraph 3.

In [ ]:
FR["speed"] = np.concatenate([R[i]["dec"]["speed"] for i in range(len(R))])
TRIAL_SPEED = {i: R[i]["trial_speed"] for i in range(len(R))}
def supp13(s_time, speed_reading, zone_reading):
    s = s_time[s_time.silent]
    if zone_reading == "b":
        s = s[((s.state == 1) & (s.zone == 3)) | ((s.state == 2) & (s.zone == 2))]
    s = s.groupby(["session", "trial", "state", "group"], as_index=False).slope.mean()
    if speed_reading == "a":
        s["spd"] = [TRIAL_SPEED[a][b] for a, b in zip(s.session.values, s.trial.values)]
    else:
        df = FR[(FR.zone >= 0) & FR.silent]
        if zone_reading == "b":
            df = df[((df.state == 1) & (df.zone == 3)) | ((df.state == 2) & (df.zone == 2))]
        zsp = df.groupby(["session", "trial"]).speed.mean().rename("spd").reset_index()
        s = s.merge(zsp, on=["session", "trial"], how="left")
    s["fast"] = s.spd > s.groupby(["session", "state"]).spd.transform("median")
    out = []
    for g in GROUPS:
        for c in (1, 2):
            f_ = s[(s.group == g) & (s.state == c) & s.fast].slope; sl = s[(s.group == g) & (s.state == c) & ~s.fast].slope
            p, z = F.mwu(f_, sl)
            out.append(dict(group=g, condition=COND_NAMES[c], fast=np.nanmedian(f_), slow=np.nanmedian(sl), z=z, p=p, n=f"{f_.notna().sum()}, {sl.notna().sum()}"))
    return pd.DataFrame(out)
for rd, sh in DEC_CONFIGS:
    st_ = zone_slopes(col((rd, sh, "lognormal"), 1), against="time")
    for spr in "ab":
        for zr in "ab":
            t = supp13(st_, spr, zr)
            key = t.group.isin(["PPC L2", "PPC L3"])
            ok = (t.z < 0) & (t.p < 0.05)     # MAP falls faster (more negative slope on time) in fast trials
            print(f"{label(rd, sh)}, speed {spr}, zones {zr}: fast steeper P < 0.05 in PPC L2/L3 {ok[key].sum()} of 4; others {ok[~key].sum()} of 8")
            if (rd, sh) == MAIN and spr == "a" and zr == "a":
                print(t.round(5).to_string(index=False))

**Result: Not reproduced.** Fast trials have significantly steeper MAP decreases (P < 0.05) in at most
three of the four PPC layer 2 and 3 tests under any reading, and in one or two under most;
the criterion needs all four under every reading.

## lognormal-decoder-better

> The decoder with a log-normal likelihood estimated goal distance better than one with a Gaussian likelihood in 17 of 18 settings, all but PM layer 5 in the continuous condition.

Source: Online Methods, Probabilistic decoding of goal distance, paragraph 3.

In [ ]:
for rd, sh in DEC_CONFIGS:
    ln = col((rd, sh, "lognormal"), 0), np.abs(col((rd, sh, "lognormal"), 1) - FR.x.values)
    ga = col((rd, None, "gaussian"), 0), np.abs(col((rd, None, "gaussian"), 1) - FR.x.values)
    for k, name in ((0, "RMSE"), (1, "absolute MAP error")):
        out = []
        for g in GROUPS:
            for c in range(3):
                m = ((FR.group == g) & (FR.state == c)).values & ~np.isnan(ln[k])
                p, z, n = F.wilcoxon_z(ln[k][m], ga[k][m])
                out.append(dict(group=g, condition=COND_NAMES[c], lognormal=np.median(ln[k][m]), gaussian=np.median(ga[k][m]), z=z, p=p, n=n))
        t = pd.DataFrame(out)
        ok = (t.z < 0) & (t.p < 0.01)
        print(f"{label(rd, sh)}, {name}: log-normal better with P < 0.01 in {ok.sum()} of 18; Gaussian better with P < 0.01 in {((t.z > 0) & (t.p < 0.01)).sum()}")
        if (rd, sh) == MAIN:
            print(t.round(4).to_string(index=False))

**Result: Not reproduced.** The Gaussian-likelihood decoder has the lower RMSE with P < 0.01 in 15 to
18 of 18 settings, and the log-normal decoder in none. By absolute MAP error the log-normal
decoder is better in 11 or 12 of 18 settings and the Gaussian in 5 or 6, short of 16.

## Single-basis regressions

For every neuron significant in goal-aligned activity under either baseline, unfiltered dR/R
is fitted as y = b0 + b1 phi(variable), phi the Gaussian basis of equation (3) centred at the
value of maximal averaged activity (Online Methods, Regression analysis). The width is
profiled over a grid of 25 values from 0.5 to 200 (cm or dB), the least-squares fit at each,
in place of an optimizer started from four initial values; the grid covers the four initial
values of either reading of O6. Frames are those of O6, (a) and (b). The fits run in the
loading step (`funamizu.regressions`).

In [ ]:
def reg_col(model, fr_reading, k):
    """One output of the single-basis fits (0 RMSE, 1 width, 2 b1) as an array over all neurons;
    NaN for neurons that were not fitted (not goal-aligned significant under either baseline)."""
    out = np.full(len(NT), np.nan)
    for i, r in enumerate(R):
        if r.get("reg"):
            idx = [roi_index[(i, n)] for n in r["dec"]["neurons"]]
            out[idx] = r["reg"][(model, fr_reading)][k]
    return out
print("neurons fitted:", int(np.isfinite(reg_col("distance", "a", 0)).sum()))

## supp6-distance-not-sound

> Goal-distance and after-reaching neurons were fit better by goal distance than by a distance held constant during no-sound zones or by the sound intensity.

Source: Supplementary Figure 6; Results, Goal-distance representation in PPC and PM, paragraph 6.

In [ ]:
for rd in CLS_GD:
    c = CLS_GD[rd]
    for fr in "ab":
        for snd in ("sound_a", "sound_b"):
            out = []
            dist = reg_col("distance", fr, 0)
            for alt in ("step", snd):
                other = reg_col(alt, fr, 0)
                for cls in ("gd", "after"):
                    for g in GROUPS:
                        m = c[cls].values & (NT.group == g).values
                        p, z, n = F.wilcoxon_z(dist[m], other[m])
                        out.append(dict(against=alt, neurons=cls, group=g, distance=np.nanmedian(dist[m]), other=np.nanmedian(other[m]), z=z, p=p, n=n))
            t = pd.DataFrame(out)
            ok = (t.z < 0) & (t.p < 0.05)
            print(f"O1 {rd[0]} O3 {rd[2]}, frames {fr}, {snd}: Distance lower RMSE with P < 0.05 in {ok.sum()} of {len(t)}")
            if rd == ("a", "67", "a") and fr == "a" and snd == "sound_a":
                print(t.round(6).to_string(index=False))

**Result: Partially reproduced.** Distance has the lower RMSE with P < 0.05 in 23 of the 24
comparisons under every reading (frames, sound below 25 cm, classification). The exception
is PM layer 5 after-reaching neurons against Step (P = 0.10, n = 76 in the reading shown).
The differences of median RMSE are small, below 0.002 dR/R.

## tuning-similar-ppc-pm

> The distance tuning curves of goal-distance neurons, as fitted coefficients, did not differ between PPC and PM.

Source: Results, Goal-distance representation in PPC and PM, paragraph 6.

A claim of absence; the coefficient compared is open (claims file), readings (a) the width s
and (b) b1 of the distance fit, the distance-only set of equation (2).

In [ ]:
for rd in CLS_GD:
    c = CLS_GD[rd]
    for fr in "ab":
        for k, name in ((1, "width s"), (2, "coefficient b1")):
            v = reg_col("distance", fr, k)
            out = []
            for L in (2, 3, 5):
                a = v[c.gd.values & (NT.area == "PPC").values & (NT.layer == L).values]
                b = v[c.gd.values & (NT.area == "PM").values & (NT.layer == L).values]
                p, z = F.mwu(a, b)
                out.append(f"L{L} PPC {np.nanmedian(a):.4g} PM {np.nanmedian(b):.4g} z {z:.2f} P {p:.3g}")
            print(f"O1 {rd[0]} O3 {rd[2]}, frames {fr}, {name}: " + "; ".join(out))

**Result: Blocked, methods underspecified.** The readings disagree. With frames from start to goal
(O6 a), PPC and PM differ in the width in all three layers (P 2e-12 to 0.03) and in b1 in
layers 3 and 5 under every classification reading. With frames to 3 s after the goal (O6 b)
and the zero baseline, the widths do not differ in any layer (P 0.095 to 0.22), while b1
differs in all three (P 5e-10 to 0.008). As the criterion directs, the claim is blocked.

## Summary

| Claim | Outcome |
|---|---|
| session-counts | reproduced |
| neuron-counts | reproduced |
| behaviour-timing | not reproduced |
| lick-goal-correlation | partially reproduced |
| fig1c-licking-without-sound | reproduced |
| supp1-licking-speed | reproduced |
| fig1d-muscimol-correlation, fig1e-muscimol-variance, fig1fg-muscimol-no-sound-licking, supp2d-muscimol-overrun, muscimol-locomotion | blocked, data missing |
| sound-responsive-layer3 | blocked, data missing |
| neuron-classes | not reproduced |
| fig2d-task-relevant-proportions | not reproduced |
| fig2d-goal-distance-ppc-more | reproduced |
| fig2d-after-vs-before | reproduced (assumption O1 a) |
| fig2d-layer-distance-preference | reproduced (assumption O1 a) |
| fig3b-distance-over-time | blocked, too large |
| supp6-distance-not-sound | partially reproduced |
| tuning-similar-ppc-pm | blocked, methods underspecified (readings disagree) |
| fig3c-continuous-ramp, fig3c-ppc-ramp-without-sound | reproduced |
| supp8a-pm-no-sound | reproduced (assumptions O1 a, trace a) |
| supp8b-slopes-lower-without-sound | reproduced (assumption O1 a) |
| supp8c-ppc-steeper-than-pm | partially reproduced (assumption O1 a) |
| supp9-preserved-representation | reproduced (assumptions O1 a, groups a) |
| fig4-above-chance | reproduced (assumption, flat-prior chance) |
| fig4c-ppc-lower-error, fig4e-ppc-lower-uncertainty | reproduced (assumption O1 a) |
| fig4d-map-near-goal | reproduced (assumptions O1 a, 4.2 cm bins) |
| supp11-sound-reduces-error | reproduced |
| fig5a-map-follows-continuous | reproduced |
| fig5a-map-decreases-without-sound | partially reproduced |
| fig5b-map-slope-lower-without-sound, fig5cd-uncertainty-updating, fig5c-uncertainty-decreases-with-sound | reproduced |
| fig5-layer3-steeper-than-layer2, supp13-map-speed, lognormal-decoder-better | not reproduced |
| svm-correlations | blocked, too large |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: lick counts and onsets come from a lick frequency,
not lick times; the task-relevance baseline (O1), the k-means feature (O3), the shift (O7) and
the bins (O8) are readings, and where they disagree an assumption is stated in the claim's
section; k-means is pooled over sessions; the multi-basis regression (fig3b) and the SVM claims are not run, and the single-basis fits profile the width over a grid.

What was found about the dataset: it holds the paper's 94 imaging sessions of 8 mice, with
the session counts per mouse, area and layer and the 41,031 active ROIs exactly as reported,
dR/R as defined in the Methods, and the first 150 trials of each session. It does not hold the
muscimol experiment or the sound-mapping trials. The sound streams hold only the start tone,
not the task sound the description implies. Licks are stored as a capped frequency, not as
events. The start-to-reward times (9.3 s) and the share of slow trials (0.48%) differ from the
Methods (7.4 s, 2.71%).